# HUH Deduplication Methodology

This notebook explores duplicate records within the archived Harvard University Herbaria Index of Botanists data before any HUH-specific deduplication rules are moved into `pkb.models.people`.

There is no complete gold standard for HUH duplicates. The aim is therefore to build high-confidence candidate clusters, inspect them, and report proxy metrics: rule yield, evidence strength, conflict checks, and review samples.

## Setup

In [ ]:
from __future__ import annotations

import ast
import re
import unicodedata
from itertools import combinations
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from nameparser import HumanName
from rapidfuzz import fuzz

from pkb.config import settings

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

OUTPUT_DIR = Path("notebooks/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load HUH Collectors

Use the archived CSV produced from the old HUH scrape. This is source-methodology work, so it reads the preserved asset directly rather than going through the build task.

In [ ]:
huh_path = settings.assets_dir / "harvard-index" / "huh-collectors.csv"
huh = pd.read_csv(huh_path, dtype=str)

print(huh.shape)
huh.head()

## Normalise Names, Dates, And Set-Like Fields

HUH records contain several useful non-ID signals: standard names, full names, life dates, geography, herbaria, institution UUIDs, remarks, and team strings. These are useful for duplicate detection, but they need to be normalised before comparison.

In [ ]:
TITLE_WORDS = {"dr", "doctor", "prof", "professor", "sir", "dame", "mr", "mrs", "ms", "miss"}
SUFFIX_WORDS = {"jr", "sr", "ii", "iii", "iv", "phd", "md", "fba", "frs"}


def clean_scalar(value):
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    value = str(value).strip()
    if not value or value.lower() in {"nan", "[none]"}:
        return None
    return value


def strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(char for char in text if not unicodedata.combining(char))


def canonicalise_name(value):
    value = clean_scalar(value)
    if not value:
        return None

    value = strip_accents(value)
    parsed = HumanName(value)
    if parsed.last and parsed.first:
        value = " ".join(part for part in [parsed.first, parsed.middle, parsed.last] if part)
    elif "," in value:
        surname, given = [part.strip() for part in value.split(",", 1)]
        value = f"{given} {surname}"

    value = re.sub(r"[^A-Za-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).lower().strip()
    tokens = [token for token in value.split() if token not in TITLE_WORDS | SUFFIX_WORDS]
    return " ".join(tokens) or None


def parse_name(value):
    canonical = canonicalise_name(value)
    tokens = canonical.split() if canonical else []
    return pd.Series({
        "canonicalName": canonical,
        "initials": "".join(token[0] for token in tokens) or None,
        "firstToken": tokens[0] if tokens else None,
        "lastToken": tokens[-1] if tokens else None,
    })


def clean_year(value):
    value = clean_scalar(value)
    if not value:
        return pd.NA
    match = re.search(r"([+-]?\d{3,4})", value)
    if not match:
        return pd.NA
    return int(match.group(1).replace("+", ""))


def parse_collection(value):
    value = clean_scalar(value)
    if not value:
        return []
    try:
        parsed = ast.literal_eval(value)
    except (SyntaxError, ValueError):
        parsed = value
    if isinstance(parsed, (set, list, tuple)):
        return sorted({str(item).strip() for item in parsed if clean_scalar(item)})
    return [str(parsed).strip()] if clean_scalar(parsed) else []


def jaccard(a, b):
    a = set(a or [])
    b = set(b or [])
    if not a and not b:
        return np.nan
    return len(a & b) / len(a | b)

In [ ]:
work = huh.copy()
work["huhID"] = work["id"].astype(str)
work["displayName"] = work["Standard/Label Name"].combine_first(work["Full Name"]).combine_first(work["Name"])
work = pd.concat([work, work["displayName"].apply(parse_name)], axis=1)
work["standardNameKey"] = work["Standard/Label Name"].apply(canonicalise_name)
work["fullNameKey"] = work["Full Name"].apply(canonicalise_name)
work["birthYearClean"] = work["birthYear"].combine_first(work["Date of birth"]).apply(clean_year).astype("Int64")
work["deathYearClean"] = work["deathYear"].combine_first(work["Date of death"]).apply(clean_year).astype("Int64")

for column in ["geographyISO", "herbariaCode", "institutionUUID"]:
    work[column + "List"] = work[column].apply(parse_collection)

work[["huhID", "displayName", "canonicalName", "standardNameKey", "fullNameKey", "birthYearClean", "deathYearClean", "geographyISOList", "herbariaCodeList"]].head()

## Candidate Generation

Block by standard name where possible, then by surname plus life dates. This should find likely duplicates without comparing every HUH record to every other HUH record.

In [ ]:
def candidate_pairs_from_block(frame, block_column, max_block_size=200):
    rows = []
    blocked = frame[frame[block_column].notna()]
    for block_value, group in blocked.groupby(block_column):
        if len(group) < 2 or len(group) > max_block_size:
            continue
        for a, b in combinations(group.itertuples(index=False), 2):
            rows.append({
                "huhIDA": a.huhID,
                "huhIDB": b.huhID,
                "block": block_column,
                "blockValue": block_value,
            })
    return rows


candidate_rows = []
candidate_rows.extend(candidate_pairs_from_block(work, "standardNameKey"))

life_date_block = work.copy()
life_date_block["surnameLifeDatesKey"] = (
    life_date_block["lastToken"].fillna("")
    + "|"
    + life_date_block["birthYearClean"].astype(str)
    + "|"
    + life_date_block["deathYearClean"].astype(str)
)
candidate_rows.extend(candidate_pairs_from_block(life_date_block, "surnameLifeDatesKey"))

candidate_pairs = pd.DataFrame(candidate_rows).drop_duplicates(["huhIDA", "huhIDB"])
candidate_pairs.shape

## Pair Features And Rules

These rules are intentionally conservative. Different HUH IDs, GUIDs, and ASA Botanist IDs are not treated as negative evidence because HUH appears to contain duplicated person records.

In [ ]:
lookup = work.set_index("huhID")


def years_equal(a, b):
    return pd.notna(a) and pd.notna(b) and int(a) == int(b)


def pair_features(pair):
    a = lookup.loc[pair.huhIDA]
    b = lookup.loc[pair.huhIDB]
    standard_similarity = fuzz.token_sort_ratio(a.standardNameKey or "", b.standardNameKey or "")
    full_name_similarity = fuzz.token_sort_ratio(a.fullNameKey or "", b.fullNameKey or "")
    name_similarity = max(standard_similarity, full_name_similarity)

    return pd.Series({
        "nameA": a.displayName,
        "nameB": b.displayName,
        "standardNameA": a["Standard/Label Name"],
        "standardNameB": b["Standard/Label Name"],
        "fullNameA": a["Full Name"],
        "fullNameB": b["Full Name"],
        "birthYearA": a.birthYearClean,
        "birthYearB": b.birthYearClean,
        "deathYearA": a.deathYearClean,
        "deathYearB": b.deathYearClean,
        "standardNameMatch": a.standardNameKey is not None and a.standardNameKey == b.standardNameKey,
        "fullNameMatch": a.fullNameKey is not None and a.fullNameKey == b.fullNameKey,
        "birthYearMatch": years_equal(a.birthYearClean, b.birthYearClean),
        "deathYearMatch": years_equal(a.deathYearClean, b.deathYearClean),
        "nameSimilarity": name_similarity,
        "geographyOverlap": jaccard(a.geographyISOList, b.geographyISOList),
        "herbariaOverlap": jaccard(a.herbariaCodeList, b.herbariaCodeList),
        "institutionOverlap": jaccard(a.institutionUUIDList, b.institutionUUIDList),
        "urlA": a.URL,
        "urlB": b.URL,
        "guidA": a.GUID,
        "guidB": b.GUID,
        "asaBotanistIDA": a["ASA Botanist ID"],
        "asaBotanistIDB": b["ASA Botanist ID"],
    })


features = candidate_pairs.join(candidate_pairs.apply(pair_features, axis=1))
features.head()

In [ ]:
def score_huh_pair(row):
    evidence = []
    score = 0

    if row.standardNameMatch:
        score += 45
        evidence.append("same_standard_name")
    elif row.nameSimilarity >= 95:
        score += 35
        evidence.append("very_high_name_similarity")
    elif row.nameSimilarity >= 90:
        score += 25
        evidence.append("high_name_similarity")

    if row.fullNameMatch:
        score += 25
        evidence.append("same_full_name")
    if row.birthYearMatch:
        score += 25
        evidence.append("same_birth_year")
    if row.deathYearMatch:
        score += 25
        evidence.append("same_death_year")
    if pd.notna(row.herbariaOverlap) and row.herbariaOverlap >= 0.5:
        score += 10
        evidence.append("high_herbaria_overlap")
    if pd.notna(row.institutionOverlap) and row.institutionOverlap >= 0.5:
        score += 10
        evidence.append("high_institution_overlap")
    if pd.notna(row.geographyOverlap) and row.geographyOverlap > 0:
        score += 5
        evidence.append("geography_overlap")

    return pd.Series({"dedupeScore": score, "dedupeEvidence": evidence})


scored = features.join(features.apply(score_huh_pair, axis=1))
scored.sort_values("dedupeScore", ascending=False).head(25)

## Proxy Metrics

Without a gold standard, useful metrics are:

- candidate pair count by rule/threshold
- number of records affected
- cluster-size distribution
- conflicts among proposed clusters, such as incompatible life dates
- manual review precision on sampled high-, medium-, and low-score candidates

In [ ]:
threshold_rows = []
for threshold in [60, 70, 80, 90, 100, 110, 120]:
    accepted = scored[scored["dedupeScore"] >= threshold]
    graph = nx.Graph()
    graph.add_nodes_from(work["huhID"])
    graph.add_edges_from(accepted[["huhIDA", "huhIDB"]].itertuples(index=False, name=None))
    clusters = [cluster for cluster in nx.connected_components(graph) if len(cluster) > 1]

    threshold_rows.append({
        "threshold": threshold,
        "acceptedPairs": len(accepted),
        "affectedRecords": len(set(accepted["huhIDA"]) | set(accepted["huhIDB"])),
        "duplicateClusters": len(clusters),
        "largestCluster": max((len(cluster) for cluster in clusters), default=0),
    })

threshold_summary = pd.DataFrame(threshold_rows)
threshold_summary

In [ ]:
threshold = 100
accepted = scored[scored["dedupeScore"] >= threshold].copy()

graph = nx.Graph()
graph.add_nodes_from(work["huhID"])
graph.add_edges_from(accepted[["huhIDA", "huhIDB"]].itertuples(index=False, name=None))

cluster_rows = []
for index, cluster in enumerate(sorted(nx.connected_components(graph), key=lambda c: min(c)), start=1):
    if len(cluster) == 1:
        continue
    cluster_id = f"HUHDC{index:07d}"
    for huh_id in sorted(cluster):
        cluster_rows.append({"huhDuplicateClusterKey": cluster_id, "huhID": huh_id})

clusters = pd.DataFrame(cluster_rows)
clustered_records = clusters.merge(work, on="huhID", how="left")

cluster_summary = (
    clustered_records.groupby("huhDuplicateClusterKey")
    .agg(
        clusterSize=("huhID", "size"),
        names=("displayName", lambda values: sorted({str(value) for value in values if clean_scalar(value)})),
        birthYears=("birthYearClean", lambda values: sorted({str(value) for value in values.dropna()})),
        deathYears=("deathYearClean", lambda values: sorted({str(value) for value in values.dropna()})),
        urls=("URL", lambda values: sorted({str(value) for value in values if clean_scalar(value)})),
    )
    .reset_index()
)
cluster_summary["birthYearConflict"] = cluster_summary["birthYears"].apply(lambda values: len(values) > 1)
cluster_summary["deathYearConflict"] = cluster_summary["deathYears"].apply(lambda values: len(values) > 1)

cluster_summary.sort_values(["birthYearConflict", "deathYearConflict", "clusterSize"], ascending=False).head(25)

## Review Samples

Manual review is the main validation route. Sample across score bands rather than only reviewing the obvious top matches.

In [ ]:
review_columns = [
    "huhIDA", "huhIDB", "nameA", "nameB", "standardNameA", "standardNameB",
    "fullNameA", "fullNameB", "birthYearA", "birthYearB", "deathYearA", "deathYearB",
    "nameSimilarity", "geographyOverlap", "herbariaOverlap", "institutionOverlap",
    "dedupeScore", "dedupeEvidence", "urlA", "urlB",
]

high = scored[scored["dedupeScore"] >= 110].sample(min(50, (scored["dedupeScore"] >= 110).sum()), random_state=1)
medium = scored[(scored["dedupeScore"] >= 90) & (scored["dedupeScore"] < 110)].sample(min(50, ((scored["dedupeScore"] >= 90) & (scored["dedupeScore"] < 110)).sum()), random_state=2)
low = scored[(scored["dedupeScore"] >= 70) & (scored["dedupeScore"] < 90)].sample(min(50, ((scored["dedupeScore"] >= 70) & (scored["dedupeScore"] < 90)).sum()), random_state=3)

review_sample = pd.concat([high, medium, low], ignore_index=True, sort=False)
review_sample[review_columns].to_csv(OUTPUT_DIR / "huh_deduplication_review_sample.csv", index=False)
review_sample[review_columns].head(50)

## Save Methodology Outputs

In [ ]:
scored.to_parquet(OUTPUT_DIR / "huh_deduplication_candidate_pairs.parquet", index=False)
threshold_summary.to_parquet(OUTPUT_DIR / "huh_deduplication_threshold_summary.parquet", index=False)
clusters.to_parquet(OUTPUT_DIR / "huh_deduplication_clusters.parquet", index=False)
cluster_summary.to_parquet(OUTPUT_DIR / "huh_deduplication_cluster_summary.parquet", index=False)

print("candidate pairs", len(scored))
print("clusters", clusters["huhDuplicateClusterKey"].nunique() if len(clusters) else 0)